# MMVEC plotting with q2 code

Date created: 2/10/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

This script has been adapted by: https://github.com/biocore/mmvec/blob/master/mmvec/q2/_visualizers.py

In [2]:
import os
from os.path import join
import pandas as pd
import numpy as np
import qiime2
import biom
import pkg_resources
import q2templates
import matplotlib.pyplot as plt
from mmvec.heatmap import ranks_heatmap, paired_heatmaps
import seaborn as sns
import warnings

ModuleNotFoundError: No module named 'mmvec'

In [51]:
TEMPLATES = pkg_resources.resource_filename('mmvec.q2', 'assets')

In [52]:
# Uncomment group/plot intending to create
# skin_group = 'Healthy'
# skin_group = 'Acne_NL'
skin_group = 'Acne_L'

In [53]:
def _parse_heatmap_metadata_annotations(metadata_column, margin_palette):
    '''
    Transform feature or sample metadata into color vector for annotating
    margin of clustermap.
    Parameters
    ----------
    metadata_column: pd.Series of metadata for annotating plots
    margin_palette: str
        Name of color palette to use for annotating metadata
        along margin(s) of clustermap.
    Returns
    -------
    Returns vector of colors for annotating clustermap and dict mapping colors
    to classes.
    '''
    # Create a categorical palette to identify md col
    metadata_column = metadata_column.astype(str)
    col_names = sorted(metadata_column.unique())

    # Select Color palette
    if margin_palette == 'colorhelix':
        col_palette = sns.cubehelix_palette(
            len(col_names), start=2, rot=3, dark=0.3, light=0.8, reverse=True)
    else:
        col_palette = sns.color_palette(margin_palette, len(col_names))
    class_colors = dict(zip(col_names, col_palette))

    # Convert the palette to vectors that will be drawn on the matrix margin
    col_colors = metadata_column.map(class_colors)

    return col_colors, class_colors


def _parse_taxonomy_strings(taxonomy_series, level):
    '''
    taxonomy_series: pd.Series of semicolon-delimited taxonomy strings
    level: int
        taxonomic level for annotating clustermap.
     Returns
     -------
    Returns a pd.Series of taxonomy names at specified level,
        or terminal annotation
    '''
    return taxonomy_series.apply(lambda x: x.split(';')[:level][-1].strip())


def _process_microbe_metadata(ranks, microbe_metadata, level, margin_palette):
    _warn_metadata_filtering('microbe')
    microbe_metadata, ranks = microbe_metadata.align(
        ranks, join='inner', axis=0)
    # parse semicolon-delimited taxonomy
    if level > -1:
        microbe_metadata = _parse_taxonomy_strings(microbe_metadata, level)
    # map metadata categories to row colors
    row_colors, row_class_colors = _parse_heatmap_metadata_annotations(
        microbe_metadata, margin_palette)

    return microbe_metadata, ranks, row_colors, row_class_colors


def _process_metabolite_metadata(ranks, metabolite_metadata, margin_palette):
    _warn_metadata_filtering('metabolite')
    _ids = set(metabolite_metadata.index) & set(ranks.columns)
    ranks = ranks[sorted(_ids)]
    metabolite_metadata = metabolite_metadata.reindex(ranks.columns)
    # map metadata categories to column colors
    col_colors, col_class_colors = _parse_heatmap_metadata_annotations(
        metabolite_metadata, margin_palette)

    return metabolite_metadata, ranks, col_colors, col_class_colors


def _warn_metadata_filtering(metadata_type):
    warning = ('Conditional probabilities table and {0} metadata will be '
               'filtered to contain only the intersection of IDs in each. If '
               'this behavior is undesired, ensure that all {0} IDs are '
               'present in both the table and the metadata '
               'file'.format(metadata_type))
    warnings.warn(warning, UserWarning)


def _normalize_table(table, method):
    '''
    Normalize column data in a dataframe for plotting in clustermap.

    table: pd.DataFrame
        Input data.
    method: str
        Normalization method to use.

    Returns normalized table as pd.DataFrame
    '''
    if 'col' in method:
        axis = 0
    elif 'row' in method:
        axis = 1
    if 'z_score' in method:
        res = table.apply(lambda x: (x - x.mean()) / x.std(), axis=axis)
    elif 'rel' in method:
        res = table.apply(lambda x: x / x.sum(), axis=axis)
    elif method == 'log10':
        res = table.apply(lambda x: np.log10(x + 1))
    return res.fillna(0)

In [54]:
### Original q2-MMVEC heatmap function
def heatmap(output_dir: str,
            ranks: pd.DataFrame,
            microbe_metadata: pd.Series = None,
            metabolite_metadata: pd.Series = None,
            method: str = 'average',
            metric: str = 'euclidean',
            color_palette: str = 'RdBu_r',
            margin_palette: str = 'cubehelix',
            x_labels: bool = False,
            y_labels: bool = False,
            level: int = -1,
            row_center: bool = True) -> None:
    """
    Generate and save a heatmap without using QIIME's q2templates.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if microbe_metadata is not None:
        microbe_metadata = microbe_metadata.to_series()
    if metabolite_metadata is not None:
        metabolite_metadata = metabolite_metadata.to_series()
    
    ranks = ranks.T  # Transpose the dataframe

    if row_center:
        ranks = ranks - ranks.mean(axis=0)

    # Generate heatmap
    hotmap = ranks_heatmap(ranks, microbe_metadata, metabolite_metadata,
                           method, metric, color_palette, margin_palette,
                           x_labels, y_labels, level)

    # Save heatmap as PDF and PNG
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_original.pdf'), bbox_inches='tight')
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_original.png'), bbox_inches='tight')

    print(f"Heatmap saved in {output_dir}")

In [55]:
# Define the output directory
output_dir = f"MMVEC/outputs/{skin_group}/"

# Load the QIIME 2 artifact and extract the dataframe
ranks_artifact = qiime2.Artifact.load(f"MMVEC/outputs/{skin_group}/conditionals.qza")
ranks = ranks_artifact.view(pd.DataFrame)  # Convert to pandas DataFrame
ranks.index.name = None
ranks

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
Phenylalanine,1.503038,0.827910,0.480633,0.424425,1.052926,0.286075,0.162422,0.062398,0.116582
D-TRYPTOPHAN,1.249937,0.859670,0.599016,0.411111,-0.259540,0.227704,-0.023857,-0.753317,-0.113970
C19H36O4 Fatty acid,0.983855,0.956422,0.855260,0.755212,0.367408,0.701812,0.611159,0.598937,0.652526
Nicotine,0.038156,0.097065,0.070868,0.031636,-0.204879,0.013155,0.004493,0.060747,-0.005258
C19H38O4 Fatty acid,-0.422375,-0.225845,-0.113987,-0.056513,0.012995,0.004119,0.096200,0.291123,0.098498
N-Oleoylethanolamine,-0.587479,-0.464616,-0.433527,-0.433428,-0.545340,-0.417789,-0.383140,-0.250646,-0.383969
Urocanic acid,0.051801,0.331955,0.543247,0.682112,0.966255,0.786274,0.972949,1.185803,0.908892
Glutamyltyrosine,-1.277271,-1.058900,-0.824192,-0.700710,-0.288226,-0.555013,-0.476780,-0.306454,-0.309830
Sorbitane Monooleate,-0.690915,-0.625968,-0.538168,-0.495274,-0.430178,-0.457868,-0.433289,-0.515377,-0.426387
Gln-C14:0,-0.848747,-0.697694,-0.639150,-0.618570,-0.671422,-0.588469,-0.530158,-0.373214,-0.537084


In [56]:
# Run the heatmap function with the same parameters as in the qiime command to generate the aesthetically unaltered plots
heatmap(
    output_dir=output_dir,
    ranks=ranks,
    microbe_metadata=None,  # If you have metadata, load it as a pandas Series
    metabolite_metadata=None,  # If you have metabolite metadata, load it
    method="average",
    metric="euclidean",
    color_palette="RdBu_r",
    margin_palette="cubehelix",
    x_labels=True,
    y_labels=True,
    level=50,
    row_center=True
)

Heatmap saved in MMVEC/outputs/Healthy/


### Create new plots with aesthetic changes

In [57]:
def heatmap_stylized(output_dir: str,
            ranks: pd.DataFrame,
            microbe_metadata: pd.Series = None,
            metabolite_metadata: pd.Series = None,
            method: str = 'average',
            metric: str = 'euclidean',
            color_palette: str = 'RdBu_r',
            margin_palette: str = 'cubehelix',
            x_labels: bool = False,
            y_labels: bool = False,
            row_order=None,
            col_order=None,
            level: int = -1,
            row_center: bool = True) -> None:
    """
    Generate and save a heatmap without using QIIME's q2templates.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if microbe_metadata is not None:
        microbe_metadata = microbe_metadata.to_series()
    if metabolite_metadata is not None:
        metabolite_metadata = metabolite_metadata.to_series()
    
    ranks = ranks.T  # Transpose the dataframe

    if row_center:
        ranks = ranks - ranks.mean(axis=0)

    # **Subset microbe metadata based on rows/columns**
    if microbe_metadata is not None:
        microbe_metadata, ranks, row_colors, row_class_colors = _process_microbe_metadata(
            ranks, microbe_metadata, level, margin_palette)
    else:
        row_colors = None

    # **Subset metabolite metadata based on rows/columns**
    if metabolite_metadata is not None:
        metabolite_metadata, ranks, col_colors, col_class_colors = _process_metabolite_metadata(
            ranks, metabolite_metadata, margin_palette)
    else:
        col_colors = None

    # **Apply custom row & column ordering**
    if row_order is not None:
        row_order = [r for r in row_order if r in ranks.index]
        ranks = ranks.loc[row_order]

    if col_order is not None:
        col_order = [c for c in col_order if c in ranks.columns]
        ranks = ranks[col_order]

    # **Apply fixed color scale range**
    vmin = -1.75
    vmax = 1.75

    # **Generate heatmap with smaller cells**
    hotmap = sns.clustermap(ranks, cmap=color_palette,
                            col_colors=col_colors, row_colors=row_colors,
                            figsize=(6, 6),  # Reduce figure size
                            method=method, metric=metric,
                            dendrogram_ratio=(0.03, 0.03),  # Reduce dendrogram size
                            row_cluster=False, col_cluster=False,  # Dendrogram clustering
                            vmin=vmin, vmax=vmax,  # Fixed scale range
                            cbar_kws={'label': 'Log Conditional Probability',
                                      'orientation': 'horizontal',
                                      'shrink': 0.5})
    
    # **Format Colorbar**
    hotmap.cax.set_position([0.25, -0.1, 0.5, 0.02])  # Move colorbar down
    cbar = hotmap.ax_heatmap.collections[0].colorbar
    cbar.set_label("Log Conditional Probability", fontsize=10, labelpad=10)
    cbar.ax.tick_params(labelsize=10)

    # ** Add title**
    titles = {
        'Acne_L': 'Acne Lesional',
        'Acne_NL': 'Acne Non-Lesional',
        'Healthy': 'Healthy'
    }
    plt.suptitle(titles.get(skin_group, ''), fontsize=16, y=1.0)

    # **Increase Axis Label Sizes**
    hotmap.ax_heatmap.tick_params(axis='x', labelsize=12, rotation=90)
    hotmap.ax_heatmap.tick_params(axis='y', labelsize=12)

    # Save heatmap as PNG and SVG
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.png'), bbox_inches='tight', dpi=600)
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.svg'), bbox_inches='tight')

    print(f"Heatmap saved in {output_dir}")


In [58]:
# Define the output directory
output_dir = f"MMVEC/outputs/{skin_group}/"

# Ensure row and column orders are the same across all figures
microbe_order = [' g__Cutibacterium', ' g__Corynebacterium', ' g__Staphylococcus', ' g__Streptococcus', ' g__Lawsonella', 
                 ' g__Micrococcus', ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus', ' g__Rothia']

metabolite_order = ['D-TRYPTOPHAN', 'Phenylalanine', 'C19H36O4 Fatty acid', 
                'C19H38O4 Fatty acid', 'N-Oleoylethanolamine', 'Urocanic acid',
                'Glutamyltyrosine', 'Sorbitane Monooleate', 'Gln-C14:0', 'Nicotine']

# Load the QIIME 2 artifact and extract the dataframe
ranks_artifact = qiime2.Artifact.load(f"MMVEC/outputs/{skin_group}/conditionals.qza")
ranks = ranks_artifact.view(pd.DataFrame)  # Convert to pandas DataFrame
ranks.index.name = None

# Reorder by microbe_order and metabolite_order
ranks = ranks.reindex(index=metabolite_order, columns=microbe_order)
# ranks = ranks.iloc[g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind]


ranks

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
D-TRYPTOPHAN,1.249937,0.859670,0.599016,0.411111,-0.259540,0.227704,-0.023857,-0.753317,-0.113970
Phenylalanine,1.503038,0.827910,0.480633,0.424425,1.052926,0.286075,0.162422,0.062398,0.116582
C19H36O4 Fatty acid,0.983855,0.956422,0.855260,0.755212,0.367408,0.701812,0.611159,0.598937,0.652526
C19H38O4 Fatty acid,-0.422375,-0.225845,-0.113987,-0.056513,0.012995,0.004119,0.096200,0.291123,0.098498
N-Oleoylethanolamine,-0.587479,-0.464616,-0.433527,-0.433428,-0.545340,-0.417789,-0.383140,-0.250646,-0.383969
Urocanic acid,0.051801,0.331955,0.543247,0.682112,0.966255,0.786274,0.972949,1.185803,0.908892
Glutamyltyrosine,-1.277271,-1.058900,-0.824192,-0.700710,-0.288226,-0.555013,-0.476780,-0.306454,-0.309830
Sorbitane Monooleate,-0.690915,-0.625968,-0.538168,-0.495274,-0.430178,-0.457868,-0.433289,-0.515377,-0.426387
Gln-C14:0,-0.848747,-0.697694,-0.639150,-0.618570,-0.671422,-0.588469,-0.530158,-0.373214,-0.537084
Nicotine,0.038156,0.097065,0.070868,0.031636,-0.204879,0.013155,0.004493,0.060747,-0.005258


In [59]:
# If trying to do explicit probabilities, see https://github.com/biocore/mmvec/issues/93
# from skbio.stats.composition import clr_inv

# Obtain the explicit probabilities from these ranks
# probs = ranks.apply(clr_inv, axis=0)

# Check column sums (should equal 1 after applying softmax transform)
# col_sums = np.sum(probs, axis=0)
# print("Row sums: ", col_sums)

In [60]:
# Run the heatmap function with to generate same MMVEC heatmap but with preferred aesthetics
heatmap_stylized(
    output_dir=output_dir,
    ranks=ranks,
    microbe_metadata=None,  # If you have metadata, load it as a pandas Series
    metabolite_metadata=None,  # If you have metabolite metadata, load it
    method="average",
    metric="euclidean",
    color_palette="RdBu_r",
    margin_palette="cubehelix",
    x_labels=True,
    y_labels=True,
    row_order=microbe_order,
    col_order=metabolite_order,
    level=50,
    row_center=True
)

Heatmap saved in MMVEC/outputs/Healthy/
